In [1]:
from hydra import initialize, compose
from opentelemetry.trace import SpanKind
from social_groups.trialrunner.utils.cli_utils import setup_config
import phoenix.otel

with initialize(version_base=None, config_path="../../configs/trials"):
    setup_config(compose(config_name="local"))

phoenix.otel.register(
    project_name="TribalCouncilTest", auto_instrument=True, verbose=False, batch=True, endpoint="http://localhost:4317",
)

In [7]:
from social_groups.trialrunner.experiment import main_registry
from social_groups.trialrunner.data_connectors.mmlu_pro_subset import MMLUProSubsetConnector
from social_groups.trialrunner.config import LLMConfig, Backend

from social_groups.trialrunner.data_connectors.mmlu_pro_single import (
    MMLUProSingleConnector,
)
from social_groups.trialrunner.decision_schemes.TribalCouncil import (
    TribalCouncilConfiguration,
    TribalCouncilDebate,
    Agent,
)
from social_groups.trialrunner.config import BackendInfo


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

data_connector = MMLUProSingleConnector()

In [10]:
debate = TribalCouncilDebate(
    TribalCouncilConfiguration(
        proposal_agent=Agent(
            llm=LLMConfig(
                max_new_tokens=4096,
                top_p=0.9,
                temperature=0.3,
                top_k=None,
                typical_p=None,
                repetition_penalty=None,
            ),
            backend=BackendInfo(
                backend=Backend.vLLMExternal, model_name="Qwen/Qwen3-0.6B"
            ),
        ),
        council_agent=Agent(
            llm=LLMConfig(
                max_new_tokens=4096,
                top_p=0.9,
                temperature=0.3,
                top_k=None,
                typical_p=None,
                repetition_penalty=None,
            ),
            backend=BackendInfo(
                backend=Backend.vLLMExternal, model_name="Qwen/Qwen3-0.6B"
            ),
        ),
        council_size=3,
    )
)

In [ ]:
example_input = data_connector.prepare_example(next(data_connector.iterate_data()))

In [ ]:
proposals = debate.proposal_round(example_input.question)
proposals


In [ ]:
from social_groups.trialrunner.utils.phoenix import phoenix_log_span
from opentelemetry import trace
from openinference.instrumentation import capture_span_context

tracer = trace.get_tracer("Test_Trace")
with capture_span_context() as capture:
    with tracer.start_as_current_span("my_span", attributes={"openinference.span.kind": "chain"}) as span:
        with phoenix_log_span("One Round Over."):
            reactions = debate.question_round(example_input.question, proposals)

reactions

In [ ]:
answers = debate.opinion_round(example_input.question, proposals, reactions)
answers

In [ ]:
q = data_connector.prepare_example(next(data_connector.iterate_data())).question
q

In [ ]:
results = q[3:].strip().split("\n")[2:]
answers = {}
for result in results:
    print(result.strip()[1:].split("):"))
    letter, word = result.strip()[1:].split("):")

    answers[letter] = word